In [ ]:
!git clone -q https://github.com/<you>/BECSimulation.git
%cd BECSimulation
%load_ext autoreload
%autoreload 2

In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from config import SimulationConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = SimulationConfig(
    # --- required: no defaults, you must state these ---
    Np=1e5,                       # particle number
    m=1.9e-25,                    # atomic mass (kg)
    omega=2 * torch.pi * 40,      # trap frequency (rad/s)
    a0=2e-9,                      # s-wave scattering length (m)
    a02=5e-23,                    # scattering volume, g2 term (m^3)
    up=2.5e-5,                    # half box length (m)
    N=256,                        # grid points in x and y

    # --- geometry ---
    Nz=None,                      # None means same as N
    sigma=10e-6,                  # width of the Gaussian ansatz (m)

    # --- trap ---
    gamma=1.0,                    # z anisotropy, real time
    imag_gamma=1.0,               # z anisotropy, imaginary time
    modulation_amp=0.1,
    modulation_freq=2 * 5**0.5,

    # --- real time ---
    ev_time=0.015,
    time_steps=10_000,

    # --- imaginary time ---
    imag_dtau=1e-4,
    imag_max_steps=10_000,
    tolerance=1e-6,

    # --- numerics ---
    cutoff="Hard Cutoff",         # see the registry in cutoff.py
    cutoff_coeff=0.9,
    include_g0=True,
    include_g2=True,
    high_precision=True,

    # --- what to record (all opt-in; see "Diagnostics" below) ---
    track_waist=True,
    track_modes=False,
)

print(cfg)                        # dataclass prints every field, labelled
print(cfg.to_dict()["_derived"])  # l, dt, G0, G2, ...

In [ ]:
from dataclasses import replace
cfg2 = replace(cfg, gamma=1.3)         # validation re-runs; derived values follow
scan = [replace(cfg, omega=w) for w in omegas]

In [ ]:
from grid import Grid
from cutoff import make_cutoff
from interactions import make_terms
from potentials import make_potential
from evolution import RealTimeEvolution, ImaginaryTimeEvolution
from diagnostics import Recorder, NormMonitor
import storage, plotting

grid   = Grid(cfg, device)
cutoff = make_cutoff(cfg, grid)
terms  = make_terms(cfg, grid, cutoff)
names  = [t.name for t in terms]

In [ ]:
path = storage.ground_state_path(cfg)

if path.exists():
    psi, meta = storage.load_ground_state(path, cfg, device)
    print(f"Loaded ground state, E = {meta['final_energy']:.6f}, "
          f"{meta['steps']} steps, created {meta['created']}")
else:
    rec_ite = Recorder.energies_only(names, dt=cfg.imag_dtau)
    ite = ImaginaryTimeEvolution(
        grid, make_potential(cfg, grid, cfg.imag_gamma), terms,
        dtau=cfg.imag_dtau, max_steps=cfg.imag_max_steps,
        recorder=rec_ite, tolerance=cfg.tolerance)
    psi = ite.run(grid.gaussian().to(device))
    storage.save_ground_state(path, psi, cfg,
                              converged=ite.converged,
                              final_energy=rec_ite.last_total(),
                              steps=ite.steps_taken)
    print(f"Computed and cached: {path}")

In [ ]:
rec = Recorder(names,
               measure_every=10,          # energies every 10th step
               dt=cfg.dt,
               track_modes=cfg.track_modes,
               track_frames=True,         # for the GIF
               radius_squared=grid.radius_squared() if cfg.track_waist else None,
               dV=grid.dV)

rte = RealTimeEvolution(
    grid, make_potential(cfg, grid, cfg.gamma), terms,
    dtau=cfg.dtau, max_steps=cfg.time_steps,
    recorder=rec, measure_every=10,
    monitors=[NormMonitor(grid)])

psi_final = rte.run(psi)

out = "storage/run_2026_09_13"
storage.save_run(out, cfg, grid, rec,
                 description="modulation at 2*sqrt(5), watching the k=2 mode",
                 psi_final=psi_final.cpu().numpy())

from google.colab import drive; drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/BECSimulation/storage"
storage.save_run(f"{OUT}/run_x", cfg, grid, rec, ...)
storage.ground_state_path(cfg, root=OUT)

In [ ]:
cfg_dict, arrays = storage.load_run(f"{out}/run.npz")

figs, gifs = plotting.standard_set(arrays, out_dir=f"{out}/plots")
figs["energies"]                                  # renders inline
#figs["energies"].axes[0].set_yscale("log")        # adjust after the fact
plotting.save_set(figs, f"{out}/plots")

# Ignore Below: Golden Test

In [27]:
%load_ext autoreload
%autoreload 2

import torch
from config import SimulationConfig 
from grid import Grid
from cutoff import make_cutoff 
from interactions import make_terms
from potentials import make_potential 
from evolution import RealTimeEvolution, ImaginaryTimeEvolution
from diagnostics import Recorder, NormMonitor
import storage, plotting

device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = SimulationConfig(
    Np=1e5, m=1.9e-25, omega=2 * torch.pi * 40,
    a0=2e-9, a02=5e-23,
    up=7.5e-6, N=32,
    sigma=10e-6, ev_time=3e-4, time_steps=200,
    cutoff="Hard Cutoff", cutoff_coeff=0.9,
)

grid   = Grid(cfg, device)
cutoff = make_cutoff(cfg, grid)
terms  = make_terms(cfg, grid, cutoff)

rec = Recorder([t.name for t in terms], measure_every=10, track_modes=cfg.track_modes, dt = cfg.dtau, radius_squared=grid.radius_squared() if cfg.track_waist else None, dV=grid.dV)

rte = RealTimeEvolution(
    grid, make_potential(cfg, grid, cfg.gamma), terms,
    dtau=cfg.dtau, max_steps=cfg.time_steps,
    recorder=rec,
    monitors=[NormMonitor(grid)])

psi_final = rte.run(grid.gaussian().to(device))
storage.save_run("storage/run_2026_09_07", cfg, grid,
                 rec, psi_final=psi_final.cpu().numpy())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


PosixPath('storage/run_2026_09_07/run.npz')

In [35]:
import numpy as np

base = np.load("../baseline_small.npz")

def compare(name, new, old, rtol=1e-12, atol=None):
    new, old = np.asarray(new), np.asarray(old)
    if atol is None:
        atol = 1e-15 * np.max(np.abs(old))   # floor relative to the array's scale
    ok = np.allclose(new, old, rtol=rtol, atol=atol)
    signif = np.abs(old) > atol              # ignore entries that are numerically zero
    worst = (np.max(np.abs(new[signif] - old[signif]) / np.abs(old[signif]))
             if signif.any() else 0.0)
    print(f"{name:12s} {'PASS' if ok else 'FAIL'}   worst rel diff = {worst:.3e} "
          f"(atol={atol:.2e}, {(~signif).sum()} near-zero entries skipped)")
    return ok

compare("psi_final", psi_final.cpu().numpy(), base["psi_final"])
compare("energies",  rec.energies(),        base["energies"])
#compare("kx",        rec.modes()[0],        base["kx"])

psi_final    PASS   worst rel diff = 1.851e-14 (atol=4.06e-17, 0 near-zero entries skipped)
energies     FAIL   worst rel diff = 7.635e-03 (atol=1.18e-14, 1 near-zero entries skipped)


False